# 27 — Silver First Capture Dry Run

## Objetivo

Validar **una sola instancia real** antes de recolectar las 48.

Instancia inicial:

`site_001__task_01__exec_01`

El objetivo no es entrenar nada. Se verifica que:

1. la tarea y el sitio estén definidos;
2. el `task_context.json` quede congelado;
3. la extensión produzca exactamente una fila para la instancia;
4. el export contenga las 46 columnas esperadas;
5. `instance_id`, `site_id`, `task_id` y `execution_id` coincidan;
6. las métricas respeten rangos básicos;
7. exista evidencia mínima para la posterior anotación LLM.

Primero ejecuta este notebook una vez. Debe quedar en `WAITING_FOR_MANUAL_CAPTURE`.

Después realiza la tarea manualmente con la extensión, guarda export y evidencia, y vuelve a ejecutar este mismo notebook.


## 1. Configuración

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 500)

PROJECT_DIR = Path("..").resolve()

REPORTS_DIR = PROJECT_DIR / "reports" / "tables"
SILVER_DIR = PROJECT_DIR / "data" / "silver"
META_DIR = SILVER_DIR / "metadata"
RAW_DIR = SILVER_DIR / "raw"
EXPORT_DIR = RAW_DIR / "extension_exports"
EVIDENCE_DIR = RAW_DIR / "evidence"

FIRST_INSTANCE_ID = "site_001__task_01__exec_01"

print("PROJECT_DIR:", PROJECT_DIR)
print("FIRST_INSTANCE_ID:", FIRST_INSTANCE_ID)


## 2. Cargar manifiesto, sitios, tareas y contrato X

In [ ]:
manifest = pd.read_csv(
    META_DIR
    / "pilot_manifest.csv"
)

sites = pd.read_csv(
    META_DIR
    / "site_registry_pilot.csv"
)

tasks = pd.read_csv(
    META_DIR
    / "task_registry_pilot.csv"
)

capture_contract = pd.read_csv(
    REPORTS_DIR
    / "silver_extension_capture_contract.csv"
)

X_NAMES = (
    capture_contract.loc[
        capture_contract[
            "role"
        ]
        ==
        "X",
        "column"
    ]
    .astype(str)
    .tolist()
)

assert len(X_NAMES) == 40

print("Manifest:", manifest.shape)
print("Sites:", sites.shape)
print("Tasks:", tasks.shape)
print("X:", len(X_NAMES))


## 3. Recuperar la primera instancia

In [ ]:
instance_rows = manifest[
    manifest[
        "instance_id"
    ]
    ==
    FIRST_INSTANCE_ID
].copy()

assert len(instance_rows) == 1

instance = instance_rows.iloc[0]

site_rows = sites[
    sites[
        "site_id"
    ]
    ==
    instance[
        "site_id"
    ]
]

task_rows = tasks[
    tasks[
        "task_id"
    ]
    ==
    instance[
        "task_id"
    ]
]

assert len(site_rows) == 1
assert len(task_rows) == 1

site = site_rows.iloc[0]
task = task_rows.iloc[0]

print("SITE:", site["site_name"])
print("URL:", site["base_url"])
print("TASK:", task["task_name"])
print()
print("Descripción:")
print(task["task_description"])
print()
print("Inicio:")
print(task["start_condition"])
print()
print("Éxito:")
print(task["success_criterion"])


## 4. Comprobación manual obligatoria antes de capturar

Antes de continuar con la ejecución real confirma manualmente:

- la URL abre correctamente;
- la tarea sigue siendo posible;
- no requiere iniciar sesión;
- no requiere datos personales reales;
- no requiere pago;
- la misma tarea puede repetirse para `exec_02`.

Si alguna condición falla, **no captures esa instancia**. Debe corregirse primero el registro del sitio/tarea.


## 5. Congelar `task_context.json`

In [ ]:
instance_evidence_dir = (
    EVIDENCE_DIR
    /
    FIRST_INSTANCE_ID
)

instance_evidence_dir.mkdir(
    parents=True,
    exist_ok=True
)

task_context_path = (
    instance_evidence_dir
    /
    "task_context.json"
)

task_context = {
    "instance_id":
        FIRST_INSTANCE_ID,

    "site_id":
        instance[
            "site_id"
        ],

    "site_name":
        site[
            "site_name"
        ],

    "base_url":
        site[
            "base_url"
        ],

    "task_id":
        instance[
            "task_id"
        ],

    "task_name":
        task[
            "task_name"
        ],

    "task_description":
        task[
            "task_description"
        ],

    "start_condition":
        task[
            "start_condition"
        ],

    "success_criterion":
        task[
            "success_criterion"
        ],

    "failure_criterion":
        task[
            "failure_criterion"
        ],

    "execution_id":
        instance[
            "execution_id"
        ],

    "applicable_H1":
        task[
            "applicable_H1"
        ],

    "applicable_H4":
        task[
            "applicable_H4"
        ],

    "applicable_H5":
        task[
            "applicable_H5"
        ],

    "applicable_H6":
        task[
            "applicable_H6"
        ]
}

task_context_path.write_text(
    json.dumps(
        task_context,
        ensure_ascii=False,
        indent=2,
        default=str
    ),
    encoding="utf-8"
)

print(
    "task_context.json:",
    "✅"
    if task_context_path.exists()
    else "❌"
)

print(
    task_context_path.relative_to(
        PROJECT_DIR
    )
)


## 6. Nombres exactos de archivos para esta instancia

In [ ]:
export_path = (
    EXPORT_DIR
    /
    f"{FIRST_INSTANCE_ID}.csv"
)

artifact_paths = {
    "task_context":
        task_context_path,

    "screenshot_before":
        instance_evidence_dir
        /
        "screenshot_before.png",

    "screenshot_after":
        instance_evidence_dir
        /
        "screenshot_after.png",

    "dom_snapshot_before":
        instance_evidence_dir
        /
        "dom_snapshot_before.json",

    "dom_snapshot_after":
        instance_evidence_dir
        /
        "dom_snapshot_after.json",

    "interaction_trace":
        instance_evidence_dir
        /
        "interaction_trace.json"
}

print("EXPORT:")
print(
    export_path.relative_to(
        PROJECT_DIR
    )
)

print("\nEVIDENCIA:")

for name, path in artifact_paths.items():

    print(
        name,
        "->",
        path.relative_to(
            PROJECT_DIR
        )
    )


## 7. Qué hacer durante la captura

Para `site_001__task_01__exec_01`:

1. abre el sitio indicado;
2. deja la interfaz exactamente en la `start_condition`;
3. guarda `screenshot_before.png`;
4. inicia la captura de la extensión;
5. realiza **solo** la tarea indicada;
6. detén la captura al alcanzar el `success_criterion` o `failure_criterion`;
7. guarda `screenshot_after.png`;
8. exporta la fila agregada con el nombre exacto mostrado;
9. guarda el `interaction_trace.json` bruto si la extensión lo produce;
10. si puedes exportar snapshots DOM antes/después, guárdalos con los nombres indicados.

No edites manualmente las X del CSV para “corregir” resultados.


## 8. Contrato esperado del export

In [ ]:
CONTEXT_COLUMNS = [
    "instance_id",
    "site_id",
    "task_id",
    "execution_id",
    "platform",
    "page_type"
]

REQUIRED_COLUMNS = (
    CONTEXT_COLUMNS
    +
    X_NAMES
)

print(
    "Columnas requeridas:",
    len(
        REQUIRED_COLUMNS
    )
)

assert len(REQUIRED_COLUMNS) == 46


## 9. Reglas básicas de rango

In [ ]:
RATIO_COLUMNS = [
    "feedback_coverage_ratio",
    "component_style_consistency_ratio",
    "css_class_consistency_ratio",
    "navigation_consistency_ratio",
    "label_consistency_ratio",
    "semantic_role_consistency_ratio",
    "dom_structure_similarity",
    "navigation_persistence_ratio",
    "accessible_name_ratio",
    "visible_labels_ratio",
    "visible_options_ratio"
]

BINARY_COLUMNS = [
    "feedback_present",
    "loader_present",
    "progress_indicator_present",
    "status_message_present",
    "completion_feedback_present",
    "input_validation_present",
    "client_validation_present",
    "invalid_submission_blocked",
    "confirmation_dialog_present",
    "destructive_action_protected",
    "error_feedback_present",
    "breadcrumbs_present"
]

NONNEGATIVE_COLUMNS = [
    "task_time_s",
    "mean_response_time_ms",
    "max_response_time_ms",
    "mean_pre_action_pause_s",
    "feedback_latency_ms",
    "loader_duration_ms",
    "click_count",
    "keypress_count",
    "scroll_count",
    "backtracking_count",
    "repeated_action_count",
    "navigation_steps",
    "validation_error_count",
    "user_error_count",
    "unlabeled_controls_count",
    "navigation_depth"
]


## 10. Auditar el CSV si ya existe

In [ ]:
def audit_first_export(
    path
):

    audit = {
        "export_exists":
            path.exists(),

        "read_ok":
            False,

        "rows":
            0,

        "missing_columns":
            None,

        "extra_columns":
            None,

        "context_match":
            False,

        "ratio_violations":
            0,

        "binary_violations":
            0,

        "negative_violations":
            0,

        "x_nonmissing":
            0,

        "x_missing":
            40,

        "export_valid":
            False,

        "error":
            None
    }

    if not path.exists():
        return audit

    try:

        df = pd.read_csv(
            path
        )

    except Exception as e:

        audit[
            "error"
        ] = str(e)

        return audit

    audit[
        "read_ok"
    ] = True

    audit[
        "rows"
    ] = len(
        df
    )

    missing = sorted(
        set(
            REQUIRED_COLUMNS
        )
        -
        set(
            df.columns
        )
    )

    extra = sorted(
        set(
            df.columns
        )
        -
        set(
            REQUIRED_COLUMNS
        )
    )

    audit[
        "missing_columns"
    ] = " | ".join(
        missing
    )

    audit[
        "extra_columns"
    ] = " | ".join(
        extra
    )

    if (
        len(
            df
        )
        ==
        1
        and
        len(
            missing
        )
        ==
        0
    ):

        row = df.iloc[0]

        audit[
            "context_match"
        ] = (
            str(
                row[
                    "instance_id"
                ]
            )
            ==
            FIRST_INSTANCE_ID
            and
            str(
                row[
                    "site_id"
                ]
            )
            ==
            str(
                instance[
                    "site_id"
                ]
            )
            and
            str(
                row[
                    "task_id"
                ]
            )
            ==
            str(
                instance[
                    "task_id"
                ]
            )
            and
            str(
                row[
                    "execution_id"
                ]
            )
            ==
            str(
                instance[
                    "execution_id"
                ]
            )
        )

    ratio_violations = 0

    for col in RATIO_COLUMNS:

        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )

        ratio_violations += int(
            (
                (
                    values
                    <
                    0
                )
                |
                (
                    values
                    >
                    1
                )
            ).sum()
        )

    binary_violations = 0

    for col in BINARY_COLUMNS:

        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )

        binary_violations += int(
            (
                values.notna()
                &
                ~values.isin(
                    [
                        0,
                        1
                    ]
                )
            ).sum()
        )

    negative_violations = 0

    for col in NONNEGATIVE_COLUMNS:

        if col not in df.columns:
            continue

        values = pd.to_numeric(
            df[
                col
            ],
            errors="coerce"
        )

        negative_violations += int(
            (
                values
                <
                0
            ).sum()
        )

    audit[
        "ratio_violations"
    ] = ratio_violations

    audit[
        "binary_violations"
    ] = binary_violations

    audit[
        "negative_violations"
    ] = negative_violations

    if all(
        col in df.columns
        for col in X_NAMES
    ):

        x_values = df[
            X_NAMES
        ].iloc[0]

        audit[
            "x_nonmissing"
        ] = int(
            x_values.notna().sum()
        )

        audit[
            "x_missing"
        ] = int(
            x_values.isna().sum()
        )

    audit[
        "export_valid"
    ] = (
        audit[
            "read_ok"
        ]
        and
        audit[
            "rows"
        ]
        ==
        1
        and
        len(
            missing
        )
        ==
        0
        and
        audit[
            "context_match"
        ]
        and
        ratio_violations
        ==
        0
        and
        binary_violations
        ==
        0
        and
        negative_violations
        ==
        0
    )

    return audit


export_audit = pd.DataFrame([
    audit_first_export(
        export_path
    )
])

display(
    export_audit
)


`x_missing > 0` no invalida automáticamente la instancia. Algunas métricas pueden ser `NaN` por no ser observables/aplicables. Lo importante es que la ausencia sea semánticamente correcta y no un fallo silencioso de captura.


## 11. Auditar evidencia

In [ ]:
evidence_records = []

for artifact, path in artifact_paths.items():

    evidence_records.append({
        "artifact":
            artifact,

        "exists":
            path.exists(),

        "size_bytes":
            (
                path.stat().st_size
                if path.exists()
                else 0
            ),

        "path":
            str(
                path.relative_to(
                    PROJECT_DIR
                )
            )
    })

evidence_audit = pd.DataFrame(
    evidence_records
)

display(
    evidence_audit
)


## 12. Evidencia mínima para pasar a anotación

In [ ]:
artifact_lookup = (
    evidence_audit
    .set_index(
        "artifact"
    )[
        "exists"
    ]
    .to_dict()
)

minimal_evidence = {
    "task_context":
        True,

    "screenshot_before":
        True,

    "screenshot_after":
        True,

    "interaction_trace":
        True
}

minimal_evidence_pass = all(
    artifact_lookup.get(
        artifact,
        False
    )
    ==
    required
    for artifact, required
    in minimal_evidence.items()
)

print(
    "Evidencia mínima:",
    minimal_evidence_pass
)

print(
    "DOM snapshots presentes:",
    artifact_lookup.get(
        "dom_snapshot_before",
        False
    )
    and
    artifact_lookup.get(
        "dom_snapshot_after",
        False
    )
)


Para este piloto exigimos `interaction_trace.json` como evidencia mínima porque H1/H5 pueden depender de cambios temporales o de respuesta que dos screenshots no capturan suficientemente. Los snapshots DOM son recomendables, pero se auditan por separado.


## 13. Estado del dry run

In [ ]:
export_valid = bool(
    export_audit.loc[
        0,
        "export_valid"
    ]
)

if (
    not export_path.exists()
):

    dry_run_status = (
        "WAITING_FOR_MANUAL_CAPTURE"
    )

elif (
    not export_valid
):

    dry_run_status = (
        "EXPORT_INVALID"
    )

elif (
    not minimal_evidence_pass
):

    dry_run_status = (
        "EVIDENCE_INCOMPLETE"
    )

else:

    dry_run_status = (
        "READY_FOR_LLM_LABELING"
    )

dry_run_summary = pd.DataFrame([
    {
        "instance_id":
            FIRST_INSTANCE_ID,

        "export_exists":
            export_path.exists(),

        "export_valid":
            export_valid,

        "minimal_evidence_pass":
            minimal_evidence_pass,

        "x_nonmissing":
            int(
                export_audit.loc[
                    0,
                    "x_nonmissing"
                ]
            ),

        "x_missing":
            int(
                export_audit.loc[
                    0,
                    "x_missing"
                ]
            ),

        "dry_run_status":
            dry_run_status
    }
])

display(
    dry_run_summary
)


## 14. Guardar auditoría

In [ ]:
export_audit.to_csv(
    REPORTS_DIR
    / "silver_first_capture_export_audit.csv",
    index=False
)

evidence_audit.to_csv(
    REPORTS_DIR
    / "silver_first_capture_evidence_audit.csv",
    index=False
)

dry_run_summary.to_csv(
    REPORTS_DIR
    / "silver_first_capture_status.csv",
    index=False
)

for filename in [
    "silver_first_capture_export_audit.csv",
    "silver_first_capture_evidence_audit.csv",
    "silver_first_capture_status.csv"
]:

    path = (
        REPORTS_DIR
        /
        filename
    )

    print(
        filename,
        "✅"
        if path.exists()
        else "❌"
    )


## Punto de detención

### Primera ejecución

El estado esperado es:

```text
WAITING_FOR_MANUAL_CAPTURE
```

### Después de la captura

Vuelve a ejecutar el notebook. Los estados posibles son:

- `EXPORT_INVALID`: revisar el CSV antes de continuar.
- `EVIDENCE_INCOMPLETE`: completar los artefactos faltantes.
- `READY_FOR_LLM_LABELING`: la primera instancia está lista para probar las tres anotaciones Silver.

**No captures todavía las 48 instancias.** Primero debemos validar científicamente una instancia completa `X + evidencia + Y`.
